# PyTorch implementation of CNN

## Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import time
import torch
import tracemalloc

## Net architecture

In [2]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=3, padding=1, bias=False),
            nn.MaxPool2d(2),
            nn.Conv2d(6, 16, kernel_size=3, padding=1, bias=False),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 84),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(84, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

net = Net()

In [3]:
transform = transforms.Compose([transforms.ToTensor()])

train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

loader = DataLoader(train_data, batch_size=1, shuffle=True)
x1, y1 = next(iter(loader))

net.eval()
with torch.no_grad():
    y1hat = net(x1)
    pred = y1hat.argmax(dim=1)

print(torch.cat((pred.view(-1, 1), y1.view(-1, 1)), dim=1))

tensor([[5, 4]])


In [4]:
criterion = nn.CrossEntropyLoss()

def loss_and_accuracy(model, data_loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for x, y in data_loader:
            y_hat = model(x)
            loss = criterion(y_hat, y)
            total_loss += loss.item() * x.size(0)
            
            pred = y_hat.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    avg_loss = total_loss / total
    acc = round(100 * correct / total, 2)
    return avg_loss, acc

eval_test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)
test_loss, test_acc = loss_and_accuracy(net, eval_test_loader)
print(f"loss: {test_loss:.4f}, acc: {test_acc:.2f}")

loss: 2.3052, acc: 6.88


## Learning loop

In [51]:
settings = {
    "eta": 1e-2,
    "epochs": 3,
    "batchsize": 10
}

train_loader = DataLoader(train_data, batch_size=settings["batchsize"], shuffle=True)
eval_train_loader = DataLoader(train_data, batch_size=1000, shuffle=False)

optimizer = optim.SGD(net.parameters(), lr=settings["eta"])

train_log = []
accuracy = torch.zeros((settings["epochs"], 2))

for epoch in range(settings["epochs"]):
    start_time = time.time()
    
    tracemalloc.start() 
    
    net.train()
    for x, y in train_loader:
        optimizer.zero_grad()
        y_hat = net(x)
        loss = criterion(y_hat, y)
        loss.backward()
        optimizer.step()
        
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    elapsed = time.time() - start_time
    
    peak_mem_mb = peak_mem / (1024 * 1024)
    
    loss_val, acc_val = loss_and_accuracy(net, eval_train_loader)
    test_loss, test_acc = loss_and_accuracy(net, eval_test_loader)
    
    print(f"Epoch {epoch+1} | Time: {elapsed:.2f}s | Peak Mem: {peak_mem_mb:.2f} MB | acc: {acc_val} | test_acc: {test_acc}")
    
    train_log.append({
        "epoch": epoch + 1,
        "loss": loss_val,
        "acc": acc_val,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "time_s": elapsed,
        "peak_mem_mb": peak_mem_mb
    })
    
    accuracy[epoch, 0] = acc_val
    accuracy[epoch, 1] = test_acc

Epoch 1 | Time: 16.11s | Peak Mem: 2.36 MB | acc: 83.49 | test_acc: 82.82
Epoch 2 | Time: 15.59s | Peak Mem: 2.35 MB | acc: 86.61 | test_acc: 85.91
Epoch 3 | Time: 15.52s | Peak Mem: 2.35 MB | acc: 87.27 | test_acc: 86.35


## Test

In [ ]:
import time
import csv
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

def run_benchmark(trials=5, filename="pytorch_benchmark.csv"):
    settings = {
        "eta": 1e-2,
        "epochs": 3,
        "batchsize": 10
    }

    eval_train_loader = DataLoader(train_data, batch_size=1000, shuffle=False)
    eval_test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)
    
    results = []

    for trial in range(trials):
        print(f"--- Rozpoczynam próbę {trial + 1}/{trials} ---")
        
        net = Net()
        optimizer = optim.SGD(net.parameters(), lr=settings["eta"])
        
        train_loader = DataLoader(train_data, batch_size=settings["batchsize"], shuffle=True)
        
        trial_time = 0.0
        final_train_acc = 0.0
        final_test_acc = 0.0

        for epoch in range(settings["epochs"]):
            start_time = time.time()
        
            net.train()
            for x, y in train_loader:
                optimizer.zero_grad()
                y_hat = net(x)
                loss = criterion(y_hat, y)
                loss.backward()
                optimizer.step()
                
            elapsed = time.time() - start_time
            trial_time += elapsed

            net.eval() 
            with torch.no_grad(): 
                _, final_train_acc = loss_and_accuracy(net, eval_train_loader)
                _, final_test_acc = loss_and_accuracy(net, eval_test_loader)
            
            print(f"  Epoch {epoch+1} | Train Time: {elapsed:.2f}s | Train Acc: {final_train_acc:.4f} | Test Acc: {final_test_acc:.4f}")

        print(f"> Próba {trial+1} zakończona | Całkowity czas treningu: {trial_time:.2f}s | Final Train Acc: {final_train_acc:.4f} | Final Test Acc: {final_test_acc:.4f}\n")
        
        # Do CSV zapisujemy tylko wynik z ostatniej (trzeciej) epoki, tak jak chciałeś
        results.append({
            "Trial": trial + 1,
            "Czas (s)": round(trial_time, 2),
            "Train Acc (%)": round(final_train_acc, 2),
            "Test Acc (%)": round(final_test_acc, 2)
        })

    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        fieldnames = ["Trial", "Czas (s)", "Train Acc (%)", "Test Acc (%)"]
        writer = csv.DictWriter(file, fieldnames=fieldnames, delimiter=';')
        
        writer.writeheader()
        writer.writerows(results)

    print(f"Wyniki zapisano do pliku: {filename}")

# Uruchomienie testu
run_benchmark(trials=5)

--- Rozpoczynam próbę 1/5 ---
  Epoch 1 | Train Time: 11.28s | Train Acc: 84.0500 | Test Acc: 83.2200
  Epoch 2 | Train Time: 10.99s | Train Acc: 86.6400 | Test Acc: 85.7400
  Epoch 3 | Train Time: 10.67s | Train Acc: 86.9300 | Test Acc: 85.7500
> Próba 1 zakończona | Całkowity czas treningu: 32.94s | Final Train Acc: 86.9300 | Final Test Acc: 85.7500

--- Rozpoczynam próbę 2/5 ---
  Epoch 1 | Train Time: 10.81s | Train Acc: 83.4000 | Test Acc: 82.3700
  Epoch 2 | Train Time: 10.96s | Train Acc: 86.0800 | Test Acc: 85.0000
  Epoch 3 | Train Time: 10.75s | Train Acc: 87.7400 | Test Acc: 86.7900
> Próba 2 zakończona | Całkowity czas treningu: 32.52s | Final Train Acc: 87.7400 | Final Test Acc: 86.7900

--- Rozpoczynam próbę 3/5 ---
  Epoch 1 | Train Time: 10.92s | Train Acc: 81.4000 | Test Acc: 80.7500
  Epoch 2 | Train Time: 10.66s | Train Acc: 86.3100 | Test Acc: 85.0300
  Epoch 3 | Train Time: 10.89s | Train Acc: 87.4700 | Test Acc: 86.0100
> Próba 3 zakończona | Całkowity czas trenin